In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage

import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

load_dotenv()

model = init_chat_model(
    model="deepseek-chat"
)

@tool
def get_beijing_weather() -> str:
    """Get the current weather in beijing."""
    return "20 degrees Celsius"

@tool
def get_shanghai_weather() -> str:
    """Get the current weather in shanghai."""
    return "25 degrees Celsius"

# SQLite是一个轻量级、磁盘级且不需要服务器的数据库引擎
conn = sqlite3.connect("data.sqlite", check_same_thread=False)
checkpointer = SqliteSaver(conn)

agent = create_agent(
    model=model,
    tools=[get_beijing_weather, get_shanghai_weather],
    system_prompt=SystemMessage("You are a helpful assistant."),
    checkpointer=checkpointer
)

config = {"configurable": {"thread_id": "1"}}

response1 = agent.invoke(
    {"messages": [HumanMessage("What is the weather in beijing and shanghai?")]},
    config=config
)

response2 = agent.invoke(
    {"messages": [HumanMessage("What is my previous question?")]},
    config=config
)

for message in response1["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is the weather in beijing and shanghai?
================================== Ai Message ==================================

I'll get the current weather for both Beijing and Shanghai for you.
Tool Calls:
  get_beijing_weather (call_00_6EcSoGCsD3kOz6E7pOOYcXA1)
 Call ID: call_00_6EcSoGCsD3kOz6E7pOOYcXA1
  Args:
  get_shanghai_weather (call_01_9V7QNskQhNGdwXKcwBUQkOwj)
 Call ID: call_01_9V7QNskQhNGdwXKcwBUQkOwj
  Args:
================================= Tool Message =================================
Name: get_beijing_weather

20 degrees Celsius
================================= Tool Message =================================
Name: get_shanghai_weather

25 degrees Celsius
================================== Ai Message ==================================

Here are the current weather conditions:

- **Beijing**: 20°C
- **Shanghai**: 25°C

Shanghai is currently warmer than Beijing by 5 degrees Celsius.


In [2]:
for message in response2["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is the weather in beijing and shanghai?
================================== Ai Message ==================================

I'll get the current weather for both Beijing and Shanghai for you.
Tool Calls:
  get_beijing_weather (call_00_6EcSoGCsD3kOz6E7pOOYcXA1)
 Call ID: call_00_6EcSoGCsD3kOz6E7pOOYcXA1
  Args:
  get_shanghai_weather (call_01_9V7QNskQhNGdwXKcwBUQkOwj)
 Call ID: call_01_9V7QNskQhNGdwXKcwBUQkOwj
  Args:
================================= Tool Message =================================
Name: get_beijing_weather

20 degrees Celsius
================================= Tool Message =================================
Name: get_shanghai_weather

25 degrees Celsius
================================== Ai Message ==================================

Here are the current weather conditions:

- **Beijing**: 20°C
- **Shanghai**: 25°C

Shanghai is currently warmer than Beijing by 5 degrees Celsius.
=========